# Submission Overview

This notebook solves two modeling tasks using NYC bike trip data from 2015 and 2016. The first task builds a regression model to predict trip duration. The second task extends the Naive Bayes idea to a continuous variable by modeling positive bike speed distributions and using them to estimate rider gender.

The workflow is organized as follows:

1. Load and preview the bike trip data.
2. Engineer time, distance, and calendar features for trip-duration regression.
3. Train and evaluate a regression model with a proper train/test split.
4. Create an optional pydeck map with fictional trips and model-based duration estimates.
5. Model bike speed as a continuous log-normal distribution for each gender class.
6. Learn distribution parameters from the training set and evaluate predictions on a validation/test set.


### Initial Data Preview Explanation

This first query reads the raw parquet files and converts the original string timestamps into timestamp columns. It also calculates `duration_min`, which is the trip duration in minutes. The `LIMIT 3` output is only a quick sanity check to confirm that the data loads correctly and that the timestamp parsing works.

In [ ]:
%%duckdb

-- Step 1: Load a small preview of the raw trip data and parse timestamp columns.
-- This confirms that the data source, date parsing, and duration calculation work.


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
LIMIT 3

## Problem 1: Trip duration

### Part 1: Build a Regression Model

Build a regression to predict trip duration by using
- Day of time
- Distance between start and end stations (there might be more than one way to measure it)
- Hour of day
- Weekend indicator
- Don't forget to model bias (this one is intentionally not used in lecture)
- Also any thing you want to end

### Part 2: Experiment Design

- Ensure that you properly design your experiment to report unbiased performance metric you choose

### Part 3 [Optional]: Visualize

- Generate some fictional pickup and dropoff locations for bike trips (random pair selection)
- Estimate trip duration for those say 10 trips
- Visualize them on map using `pydeck` by using redish color for slower trips and greener for faster trips.

### Problem 1 Method Explanation

For the trip-duration regression task, the target variable is `duration_min`. The main predictors are the haversine distance between start and end stations, hour of day, day of week, month, and a weekend indicator. Cyclical variables such as hour and day of week are encoded with sine and cosine transformations because time wraps around: for example, hour 23 and hour 0 are close to each other.

The data is sorted by `start_at`, and the first 80% is used for training while the final 20% is used for testing. This time-based split avoids evaluating the model on past data after training on future data. The model is compared against a simple median-duration baseline using MAE, RMSE, and R?.

In [ ]:
# Problem 1: Trip-duration regression pipeline.
# This cell loads trip data, engineers features, trains a Ridge regression model,
# and evaluates it against a simple median-duration baseline.

import duckdb
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    lat1 = np.radians(lat1.astype(float))
    lon1 = np.radians(lon1.astype(float))
    lat2 = np.radians(lat2.astype(float))
    lon2 = np.radians(lon2.astype(float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))

# Load a sample of clean trip records with valid timestamps and station coordinates.
trips = duckdb.sql("""
    SELECT
        try_strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
        try_strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at,
        "start station latitude" AS start_lat,
        "start station longitude" AS start_lon,
        "end station latitude" AS end_lat,
        "end station longitude" AS end_lon
    FROM read_parquet('s3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet')
    WHERE starttime IS NOT NULL
      AND stoptime IS NOT NULL
      AND "start station latitude" IS NOT NULL
      AND "start station longitude" IS NOT NULL
      AND "end station latitude" IS NOT NULL
      AND "end station longitude" IS NOT NULL
    LIMIT 250000
""").df()

trips = trips.dropna(subset=['start_at', 'stop_at']).copy()
# Create the regression target and the main distance feature.
trips['duration_min'] = (trips['stop_at'] - trips['start_at']).dt.total_seconds() / 60.0
trips['trip_distance_km'] = haversine_km(trips['start_lat'], trips['start_lon'], trips['end_lat'], trips['end_lon'])
trips = trips[(trips['duration_min'] > 1) & (trips['duration_min'] < 180) & (trips['trip_distance_km'] >= 0)].copy()

# Add time/calendar features requested in the assignment.
trips['start_hour'] = trips['start_at'].dt.hour
trips['start_dow'] = trips['start_at'].dt.dayofweek
trips['is_weekend'] = (trips['start_dow'] >= 5).astype(int)
trips['month'] = trips['start_at'].dt.month

# Encode cyclical time features with sine/cosine transformations.
trips['hour_sin'] = np.sin(2 * np.pi * trips['start_hour'] / 24.0)
trips['hour_cos'] = np.cos(2 * np.pi * trips['start_hour'] / 24.0)
trips['dow_sin'] = np.sin(2 * np.pi * trips['start_dow'] / 7.0)
trips['dow_cos'] = np.cos(2 * np.pi * trips['start_dow'] / 7.0)
trips['month_sin'] = np.sin(2 * np.pi * (trips['month'] - 1) / 12.0)
trips['month_cos'] = np.cos(2 * np.pi * (trips['month'] - 1) / 12.0)

feature_cols = [
    'trip_distance_km',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'month_sin',
    'month_cos',
    'is_weekend',
 ]

# Use a time-based split so the test set represents future trips.
trips = trips.sort_values('start_at').reset_index(drop=True)
split_idx = int(len(trips) * 0.8)
train_df = trips.iloc[:split_idx]
test_df = trips.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = np.log1p(train_df['duration_min'])
X_test = test_df[feature_cols]
y_test = test_df['duration_min']

# Fit a regularized linear model. fit_intercept=True includes the bias term.
model = Pipeline([
    ('scale', StandardScaler()),
    ('regressor', Ridge(alpha=1.0, fit_intercept=True))
 ])

model.fit(X_train, y_train)
predicted_minutes = np.expm1(model.predict(X_test))

mae = mean_absolute_error(y_test, predicted_minutes)
rmse = root_mean_squared_error(y_test, predicted_minutes)
r2 = r2_score(y_test, predicted_minutes)

# Compare against a simple baseline that always predicts the train median duration.
baseline = np.repeat(train_df['duration_min'].median(), len(y_test))
baseline_mae = mean_absolute_error(y_test, baseline)

print(f'rows used: {len(trips):,}')
print(f'test MAE: {mae:.2f} minutes')
print(f'test RMSE: {rmse:.2f} minutes')
print(f'test R^2: {r2:.3f}')
print(f'baseline MAE: {baseline_mae:.2f} minutes')

preview = test_df[['duration_min']].head(10).copy()
preview['predicted_min'] = predicted_minutes[:10]
preview['abs_error'] = np.abs(preview['duration_min'] - preview['predicted_min'])
preview.rename(columns={'duration_min': 'actual_min'})

### Optional Visualization

The following optional cell creates 10 fictional trips by randomly pairing station coordinates. It estimates each trip duration with the trained regression model and visualizes the routes using `pydeck`: greener lines represent faster predicted trips, while redder lines represent slower predicted trips.

### Optional Map Explanation

This optional section creates 10 fictional trips by randomly pairing station coordinates from the dataset. The trained regression model estimates the duration of each fictional trip. The routes are then displayed with `pydeck`: lines show the start-to-end connection, green points mark starts, red points mark ends, and the line color moves from green to red as predicted duration gets slower.

In [ ]:
# Problem 1 optional map: visualize fictional trips with model-estimated durations.
# The lines are straight start-to-end connections, not routed street paths.

import pandas as pd
import pydeck as pdk

# Generate 10 fictional bike trips by randomly pairing stations from the real data.
stations = pd.concat([
    trips[['start_lat', 'start_lon']].rename(columns={'start_lat': 'lat', 'start_lon': 'lon'}),
    trips[['end_lat', 'end_lon']].rename(columns={'end_lat': 'lat', 'end_lon': 'lon'})
], ignore_index=True).drop_duplicates().dropna()

sample_starts = stations.sample(10, random_state=42).reset_index(drop=True)
sample_ends = stations.sample(10, random_state=24).reset_index(drop=True)

fictional_trips = pd.DataFrame({
    'start_lat': sample_starts['lat'],
    'start_lon': sample_starts['lon'],
    'end_lat': sample_ends['lat'],
    'end_lon': sample_ends['lon'],
})

# Calculate distance for each fictional trip using the same feature logic as the model.
fictional_trips['trip_distance_km'] = haversine_km(
    fictional_trips['start_lat'],
    fictional_trips['start_lon'],
    fictional_trips['end_lat'],
    fictional_trips['end_lon']
)

# Use one fixed example time so the fictional trips differ mainly by location/distance.
fictional_trips['start_hour'] = 17
fictional_trips['start_dow'] = 2
fictional_trips['is_weekend'] = 0
fictional_trips['month'] = 6

fictional_trips['hour_sin'] = np.sin(2 * np.pi * fictional_trips['start_hour'] / 24.0)
fictional_trips['hour_cos'] = np.cos(2 * np.pi * fictional_trips['start_hour'] / 24.0)
fictional_trips['dow_sin'] = np.sin(2 * np.pi * fictional_trips['start_dow'] / 7.0)
fictional_trips['dow_cos'] = np.cos(2 * np.pi * fictional_trips['start_dow'] / 7.0)
fictional_trips['month_sin'] = np.sin(2 * np.pi * (fictional_trips['month'] - 1) / 12.0)
fictional_trips['month_cos'] = np.cos(2 * np.pi * (fictional_trips['month'] - 1) / 12.0)

# Predict trip duration for the fictional trips using the trained regression pipeline.
fictional_trips['predicted_min'] = np.expm1(model.predict(fictional_trips[feature_cols]))
fictional_trips['predicted_min_round'] = fictional_trips['predicted_min'].round(1)

# Green = faster trips, red = slower trips.
min_pred = fictional_trips['predicted_min'].min()
max_pred = fictional_trips['predicted_min'].max()
scaled = (fictional_trips['predicted_min'] - min_pred) / (max_pred - min_pred + 1e-9)
fictional_trips['color'] = scaled.apply(lambda x: [int(255 * x), int(180 * (1 - x)), 40, 190])

fictional_trips['path'] = fictional_trips.apply(
    lambda row: [[row['start_lon'], row['start_lat']], [row['end_lon'], row['end_lat']]],
    axis=1
)
fictional_trips['start_point'] = fictional_trips.apply(
    lambda row: [row['start_lon'], row['start_lat']],
    axis=1
)
fictional_trips['end_point'] = fictional_trips.apply(
    lambda row: [row['end_lon'], row['end_lat']],
    axis=1
)

# Draw trip connections and start/end markers on the map.
line_layer = pdk.Layer(
    'PathLayer',
    data=fictional_trips,
    get_path='path',
    get_color='color',
    get_width=45,
    width_min_pixels=3,
    pickable=True
)

start_layer = pdk.Layer(
    'ScatterplotLayer',
    data=fictional_trips,
    get_position='start_point',
    get_color=[40, 180, 90, 220],
    get_radius=80,
    pickable=True
)

end_layer = pdk.Layer(
    'ScatterplotLayer',
    data=fictional_trips,
    get_position='end_point',
    get_color=[220, 60, 60, 220],
    get_radius=80,
    pickable=True
)

view_state = pdk.ViewState(
    latitude=float(fictional_trips[['start_lat', 'end_lat']].mean().mean()),
    longitude=float(fictional_trips[['start_lon', 'end_lon']].mean().mean()),
    zoom=11,
    pitch=0
)

pdk.Deck(
    layers=[line_layer, start_layer, end_layer],
    initial_view_state=view_state,
    tooltip={'text': 'Predicted duration: {predicted_min_round} min'}
)

### Problem 1 Answer Summary

The regression model predicts trip duration using trip distance, hour of day, day of week, month, and weekend indicator. Distance is calculated with the haversine formula from start and end station coordinates. Cyclical variables such as hour, day of week, and month are represented with sine/cosine transformations so the model understands that values such as 23:00 and 00:00 are close to each other.

The experiment is designed with a time-based train/test split: the first 80% of trips by start time are used for training and the final 20% are used for testing. This avoids using future trips to predict earlier trips. Performance is reported with MAE, RMSE, and R?, and the model is compared against a median-duration baseline. The model improves MAE compared with the baseline, so it learns useful signal from the selected features.

### Problem 2: Extending Naive Bayesian

### Part 1: Expand the NB Regression Idea to continous variable

$$
P(gender = a, speed_{bike} = x) = P(gender = a) P(speed_{bike} = x | gender = a)
$$

- Note that $P(speed_{bike} = x | gender = a)$ is  continous distribution.
- Expand the idea
- Build a predictive model for estimation biker gender using the bike speed ?

### Part 2: Use Visualization to decide best distribution 

- How should be $P(speed_{bike} = x | gender = a)$ modeled

### Problem 2 Method Explanation

For Naive Bayes with a continuous feature, `P(speed = x | gender = a)` is represented by a probability density instead of a discrete probability. A plain Gaussian distribution is not ideal for modeling bike speed directly, because speed is strictly positive while a Gaussian distribution can assign density to negative values.

For this reason, this notebook models `speed_kph | gender` with a log-normal distribution. Equivalently, for each gender class, `log(speed_kph)` is modeled as a Gaussian distribution. This keeps the support of the speed variable positive and better matches the right-skewed shape that speed distributions often have.

The training set is used to learn one log-mean and one log-standard deviation for each gender class. Then, for each test example, the model computes the class-conditional log-normal density and combines it with the class prior to obtain posterior probabilities for gender prediction.


In [ ]:
# Problem 2: Manual log-normal Naive Bayes using bike speed as a continuous feature.
# This cell learns class-conditional positive speed distributions from the train set
# and evaluates posterior-probability predictions on the test set.

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split


def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    lat1 = np.radians(lat1.astype(float))
    lon1 = np.radians(lon1.astype(float))
    lat2 = np.radians(lat2.astype(float))
    lon2 = np.radians(lon2.astype(float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))


# Log-normal log-density used for P(speed=x | gender=a).
# This is more appropriate than a plain Gaussian because speed cannot be negative.
def lognormal_log_pdf(x, log_mean, log_std):
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 1e-12, None)
    log_std = max(float(log_std), 1e-9)
    z = (np.log(x) - log_mean) / log_std
    return -np.log(x) - np.log(log_std) - 0.5 * np.log(2 * np.pi) - 0.5 * z ** 2


# Load trips with known binary gender labels and valid coordinates.
gender_trips = duckdb.sql("""
    SELECT
        try_strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
        try_strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at,
        gender,
        "start station latitude" AS start_lat,
        "start station longitude" AS start_lon,
        "end station latitude" AS end_lat,
        "end station longitude" AS end_lon
    FROM read_parquet('s3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet')
    WHERE gender IN (1, 2)
      AND starttime IS NOT NULL
      AND stoptime IS NOT NULL
      AND "start station latitude" IS NOT NULL
      AND "start station longitude" IS NOT NULL
      AND "end station latitude" IS NOT NULL
      AND "end station longitude" IS NOT NULL
    LIMIT 250000
""").df()

gender_trips = gender_trips.dropna(subset=['start_at', 'stop_at']).copy()
gender_trips['duration_min'] = (gender_trips['stop_at'] - gender_trips['start_at']).dt.total_seconds() / 60.0
gender_trips['trip_distance_km'] = haversine_km(
    gender_trips['start_lat'],
    gender_trips['start_lon'],
    gender_trips['end_lat'],
    gender_trips['end_lon']
)
gender_trips = gender_trips[(gender_trips['duration_min'] > 1) & (gender_trips['duration_min'] < 180) & (gender_trips['trip_distance_km'] > 0)].copy()
# Convert distance and duration into the continuous positive feature used by Naive Bayes.
gender_trips['speed_kph'] = gender_trips['trip_distance_km'] / (gender_trips['duration_min'] / 60.0)
gender_trips = gender_trips[(gender_trips['speed_kph'] > 0) & (gender_trips['speed_kph'] < 60)].copy()
gender_trips['log_speed_kph'] = np.log(gender_trips['speed_kph'])

gender_trips['gender_label'] = np.where(gender_trips['gender'] == 1, 'male', 'female')

# Visualize the class-conditional speed distributions before choosing the density model.
fig = px.histogram(
    gender_trips,
    x='speed_kph',
    color='gender_label',
    nbins=60,
    histnorm='probability density',
    barmode='overlay',
    opacity=0.55,
    title='Bike speed distribution by gender'
)
fig.show()

# A log-normal model means log(speed) is modeled with a Gaussian distribution.
fig_log = px.histogram(
    gender_trips,
    x='log_speed_kph',
    color='gender_label',
    nbins=60,
    histnorm='probability density',
    barmode='overlay',
    opacity=0.55,
    title='Log bike speed distribution by gender'
)
fig_log.show()

X = gender_trips[['speed_kph']]
y = gender_trips['gender'] - 1  # gender 1 -> male/class 0, gender 2 -> female/class 1

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_nb = X_train.copy()
train_nb['log_speed_kph'] = np.log(train_nb['speed_kph'])
train_nb['target'] = y_train.to_numpy()
train_nb['gender_label'] = np.where(train_nb['target'] == 0, 'male', 'female')

# Learn log-normal distribution parameters from the train set.
# P(gender=a | speed=x) is proportional to P(gender=a) * LogNormalPDF(speed=x | log_mean_a, log_std_a).
params = train_nb.groupby(['target', 'gender_label']).agg(
    count=('speed_kph', 'count'),
    mean_speed=('speed_kph', 'mean'),
    std_speed=('speed_kph', 'std'),
    median_speed=('speed_kph', 'median'),
    log_mean=('log_speed_kph', 'mean'),
    log_std=('log_speed_kph', 'std')
).reset_index()
params['learned_prior'] = params['count'] / len(train_nb)

# The original data is highly imbalanced, so the learned prior makes the model predict
# almost everyone as male. For a fair validation of the speed distribution itself,
# use equal class priors while keeping distribution parameters learned from the train set.
params['model_prior'] = 1 / params['target'].nunique()

log_scores = pd.DataFrame(index=X_test.index)
for _, row in params.iterrows():
    target = int(row['target'])
    log_scores[target] = np.log(row['model_prior']) + lognormal_log_pdf(X_test['speed_kph'], row['log_mean'], row['log_std'])

# Normalize log posterior scores into probabilities for easier interpretation.
score_values = log_scores[[0, 1]].to_numpy()
score_values = score_values - score_values.max(axis=1, keepdims=True)
posterior = np.exp(score_values)
posterior = posterior / posterior.sum(axis=1, keepdims=True)

pred = posterior.argmax(axis=1)
proba = posterior[:, 1]

correct_count = int((pred == y_test.to_numpy()).sum())
total_count = len(y_test)

print('learned log-normal parameters from train set:')
print(params[['gender_label', 'count', 'learned_prior', 'model_prior', 'mean_speed', 'std_speed', 'median_speed', 'log_mean', 'log_std']])
print(f'correct predictions: {correct_count} / {total_count}')
print(f'accuracy: {accuracy_score(y_test, pred):.3f}')
print(f'balanced accuracy: {balanced_accuracy_score(y_test, pred):.3f}')
print(f'roc auc: {roc_auc_score(y_test, proba):.3f}')
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['male', 'female'], zero_division=0))

validation_preview = X_test.head(10).copy()
validation_preview['actual_gender'] = np.where(y_test.head(10).to_numpy() == 0, 'male', 'female')
validation_preview['p_male'] = posterior[:10, 0]
validation_preview['p_female'] = posterior[:10, 1]
validation_preview['predicted_gender'] = np.where(pred[:10] == 0, 'male', 'female')
validation_preview


### Problem 2 Answer Summary

For a continuous variable such as bike speed, Naive Bayes can be written as:

$$P(gender=a \mid speed=x) \propto P(gender=a) f_{speed \mid gender=a}(x)$$

Here, $f_{speed \mid gender=a}(x)$ is a probability density function, not a discrete probability. A plain Gaussian model is not fully appropriate for speed because it has support over the whole real line and can assign density to negative values. Since speed is always positive, this notebook uses a log-normal distribution instead. In practice, this means `log(speed_kph)` is modeled with a Gaussian distribution for each gender class.

The model learns the log-normal parameters from the train set: each gender class has its own log-mean and log-standard deviation. The original data is highly imbalanced, so the learned prior is also reported, but equal class priors are used for prediction to evaluate the speed distribution more fairly. On the validation/test set, the model calculates the posterior probability for each class and predicts the class with the larger probability. The output reports how many predictions were correct, along with accuracy, balanced accuracy, ROC AUC, confusion matrix, and a small probability preview.


### Final Validation Notes

The sanity-check cell verifies that there are no missing values in the key modeling columns, reports the original class balance, compares the model against a majority-class baseline, and prints the confusion matrix again. Balanced accuracy is especially important here because the gender classes are not evenly represented in the data.

In [ ]:
# Sanity checks for Problem 2
# These checks confirm data quality, class imbalance, and model performance
# compared with a majority-class baseline.
print('missing values in key columns:')
print(gender_trips[['speed_kph', 'gender']].isna().sum())

print('\nclass balance:')
print(gender_trips['gender_label'].value_counts())
print(gender_trips['gender_label'].value_counts(normalize=True))

majority_baseline = np.repeat(y_train.mode().iloc[0], len(y_test))
print('\nmajority baseline accuracy:', accuracy_score(y_test, majority_baseline))
print('majority baseline balanced accuracy:', balanced_accuracy_score(y_test, majority_baseline))

print('\nmanual log-normal NB confusion matrix again:')
print(confusion_matrix(y_test, pred))

print('\nmodel vs baseline check:')
print('model balanced accuracy > baseline balanced accuracy:', balanced_accuracy_score(y_test, pred) > balanced_accuracy_score(y_test, majority_baseline))
